# Question 1

Import data from CSV

In [1]:
import pandas as pd
import numpy as np
import csv

raw_data = pd.read_csv('./data.tsv', sep='\t', quoting=csv.QUOTE_NONE)

Prepare Data, by sampling and adding the label

In [2]:
def label(rating):
    if rating > 3:
        return  1
    if rating < 3:
        return 2
    if rating == 3:
        return 3

sampled = raw_data.groupby("star_rating").sample(n=50000, random_state=42)
del raw_data

dataset = pd.DataFrame()
dataset["review"] = sampled["review_body"]
dataset["star_rating"] = sampled["star_rating"]
dataset["sentiment"] = dataset["star_rating"].apply(label)

del sampled


We will perform train / test split after extracting the features from the sentences

# Question 2(a)

Import gensim and load pre-trained model

In [3]:
import gensim.downloader as api
wv = api.load('word2vec-google-news-300')

Test word embeddings and semantics

In [ ]:
# Example comparing king, man and woman. Expecting to see queen
wx = wv['king'] - wv['man'] + wv['woman']
wv.most_similar(wx, topn=5)

[('king', 0.8449392318725586),
 ('queen', 0.7300517559051514),
 ('monarch', 0.645466148853302),
 ('princess', 0.6156251430511475),
 ('crown_prince', 0.5818676352500916)]

In [21]:
# Example comparing boy, man and puppy. Expecting to see dog
wx = wv['man'] - wv['boy'] + wv['puppy']
wv.most_similar(wx, topn=5)

[('puppy', 0.7963186502456665),
 ('dog', 0.730284571647644),
 ('pooch', 0.6746339201927185),
 ('puppies', 0.6635603904724121),
 ('cat', 0.659332275390625)]

# Question 2(b)

Create model from reviews

In [18]:
from gensim import utils
import gensim.models

class Corpus:
    def __iter__(self):
        for sentence in dataset["review"].tolist():
            yield utils.simple_preprocess(str(sentence))

sentences = Corpus()
model = gensim.models.Word2Vec(sentences=sentences, vector_size=300, window=11)

Retry examples from previous part

In [19]:
wx = model.wv["king"] - model.wv["man"] + model.wv["woman"]
model.wv.most_similar(wx, topn=5)

[('spiral', 0.39033153653144836),
 ('subject', 0.3795025646686554),
 ('spine', 0.3788004517555237),
 ('grid', 0.3747011721134186),
 ('hybrid', 0.3653666079044342)]

In [23]:
wx = model.wv['man'] - model.wv['boy'] + model.wv['puppy']
model.wv.most_similar(wx, topn=5)

[('man', 0.6620852947235107),
 ('woman', 0.4718238115310669),
 ('lady', 0.4469282627105713),
 ('geek', 0.43407174944877625),
 ('guy', 0.4289233386516571)]

# Question 3

Preprocess sentences and create embeddings

In [55]:
import re
from bs4 import BeautifulSoup
import contractions
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def preprocess_and_vectorize(wv):
    def f(review):
        text = str(review).lower()
        text = BeautifulSoup(text, "html.parser").get_text(strip=True)
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = re.sub(r'[^a-z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        text = contractions.fix(text)
        stop = set(stopwords.words('english'))
        words = [lemmatizer.lemmatize(w) for w in word_tokenize(text) if w not in stop]
        vectors = [wv[w] if w in wv else [0]*300 for w in words]

        if(len(vectors) == 0): return [0]*300
        else: return np.mean(np.array(vectors), axis=0)
    
    return f;

dataset["feature_pretrained"] = dataset["review"].apply(preprocess_and_vectorize(wv))
dataset["feature_custom"] = dataset["review"].apply(preprocess_and_vectorize(model.wv))

/tmp/ipykernel_1783/3475461715.py:13: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  text = BeautifulSoup(text, "html.parser").get_text(strip=True)
/tmp/ipykernel_1783/3475461715.py:13: MarkupResemblesLocatorWarning: Th

Train Test split

In [56]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(dataset[["feature_pretrained", "feature_custom"]], dataset["sentiment"], test_size=0.2, random_state=42)

In [59]:
prep_input(X_train["feature_pretrained"])

: 

General testing skeleton

In [58]:
from sklearn.metrics import accuracy_score

prep_input = lambda x: pd.DataFrame(x.tolist(), index=x.index)

def test_model(model, feature, name):
    model.fit(prep_input(X_train[feature]), y_train)
    y_pred = model.predict(prep_input(X_test[feature]))
    print(f"{name} accuracy: {accuracy_score(y_test, y_pred)}")

In [44]:
from sklearn.linear_model import Perceptron
test_model(Perceptron(random_state=42), "feature_pretrained", "Perceptron with 'word2vec-google-news-300' features")
test_model(Perceptron(random_state=42), "feature_custom", "Perceptron with self trained Word2Vec features")

ValueError: setting an array element with a sequence.